# **Caso práctico: “Predicción de abandono de clientes en una empresa de telecomunicaciones”**

**Objetivo:**  Desarrollar una red neuronal simple con función de activación sigmoide para predecir si un cliente abandonará el servicio (churn) en función de sus características.

**Trabajo realizado por:** David García, Ainara Gimeno y Alejandro Moreno.


## **1. Preparar los datos**

Primero, importamos numpy  y preparamos los datos simulados. Usaremos 5 variables de entrada  y 1 de salida.

Es fundamental normalizar los datos (escalarlos entre 0 y 1) para que la red neuronal funcione correctamente, ya que las variables (como 'edad' y 'factura_promedio') tienen rangos muy diferentes.

In [ ]:
import numpy as np

# Datos simulados (X)
# [edad, meses_contratado, uso_mensual(GB), factura_promedio(€), soporte_llamado]
X_raw = np.array([
    [34, 1, 5, 50, 0],   # Cliente nuevo, poco uso -> No abandona
    [55, 48, 20, 70, 1], # Cliente leal, uso alto -> No abandona
    [23, 3, 50, 90, 8],  # Cliente nuevo, insatisfecho (mucho soporte) -> Abandona
    [40, 6, 15, 65, 4],  # Cliente reciente, problemas (soporte) -> Abandona
    [60, 120, 10, 40, 1], # Cliente muy leal, bajo uso -> No abandona
    [28, 2, 45, 85, 6]   # Cliente nuevo, alto uso, problemas -> Abandona
])

# Etiquetas (y): 1 = abandona, 0 = no
y = np.array([
    [0],
    [0],
    [1],
    [1],
    [0],
    [1]
])

# Normalización simple (Min-Max) para preparar los datos
X = X_raw / X_raw.max(axis=0)

print("Datos X (normalizados):")
print(X)
print("\nDatos y:")
print(y)

## **2. Implementar la red neuronal usando Python**

Ahora implementamos la red siguiendo la estructura de los ejemplos . Definimos la función sigmoide y su derivada, inicializamos los pesos y realizamos el entrenamiento (forward pass, backpropagation y actualización).

In [ ]:
# Función de activación sigmoide
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Derivada de la función sigmoide
def sigmoid_deriv(a):
    return a * (1 - a) # Se usa 'a' (salida de sigmoide) como en el ejemplo

# Inicializar parámetros (Pesos y Sesgos)
np.random.seed(42) # Para reproducibilidad

# Dimensiones: 5 entradas, 4 ocultas, 1 salida
input_dim = 5
hidden_dim = 4
output_dim = 1

# Capa 1 (Oculta): 5 entradas -> 4 neuronas
w1 = np.random.randn(input_dim, hidden_dim)
b1 = np.random.randn(1, hidden_dim)

# Capa 2 (Salida): 4 entradas -> 1 neurona
w2 = np.random.randn(hidden_dim, output_dim)
b2 = np.random.randn(1, output_dim)

# Parámetros de entrenamiento
lr = 0.1 # Tasa de aprendizaje
epochs = 10000

# Entrenamiento
for epoch in range(epochs):

    # --- Forward Pass ---
    # Capa 1 (Oculta)
    z1 = np.dot(X, w1) + b1
    a1 = sigmoid(z1)               (sigmoide en capa oculta)

    # Capa 2 (Salida)
    z2 = np.dot(a1, w2) + b2
    a2 = sigmoid(z2)            (sigmoide en capa salida)

    # --- Cálculo de pérdida (Error Cuadrático Medio)
    loss = np.mean((a2 - y) ** 2)

    # --- Backpropagation ---
    # Gradientes Capa 2 (Salida)
    dz2 = (a2 - y) * sigmoid_deriv(a2)
    dw2 = np.dot(a1.T, dz2)
    db2 = np.mean(dz2, axis=0)

    # Gradientes Capa 1 (Oculta)
    dz1 = np.dot(dz2, w2.T) * sigmoid_deriv(a1)
    dw1 = np.dot(X.T, dz1)
    db1 = np.mean(dz1, axis=0)

    # --- Actualizar parámetros ---
    w1 -= lr * dw1
    b1 -= lr * db1
    w2 -= lr * dw2
    b2 -= lr * db2

    # Mostrar progreso
    if epoch % 2000 == 0:
        print(f"Epoch {epoch}, Loss: {loss:.4f}")

print(f"Entrenamiento finalizado. Loss: {loss:.4f}")

## **3. Evaluar el modelo**
Evaluamos el modelo viendo sus predicciones sobre los mismos datos de entrenamiento. La salida a2 es una probabilidad (gracias a la sigmoide de salida ). Redondeamos el resultado para ver la clasificación (0 o 1).

In [ ]:
# Predicciones sobre los datos de entrenamiento
print("\n--- Evaluación del Modelo ---")
print("Predicciones (Probabilidad):")
print(a2)

# Redondear para obtener la clase (0 o 1)
predicciones = np.round(a2)
print("\nPredicciones (Clase):")
print(predicciones)

print("\nValores Reales (y):")
print(y)

# Calcular precisión
precision = np.mean(predicciones == y) * 100
print(f"\nPrecisión sobre datos de entrenamiento: {precision:.2f}%")

## **4. Probar con nuevos datos**

Finalmente, probamos el modelo con dos nuevos clientes (datos que no ha visto), siguiendo el estilo de la diapositiva 33.

Importante: Debemos aplicar la misma normalización que usamos en el entrenamiento.

In [ ]:
# Nuevos datos (X_test)
# Cliente A: [25 (edad), 2 (meses), 10 (GB), 55 (€), 1 (soporte)] -> Probablemente NO abandona
# Cliente B: [30 (edad), 5 (meses), 60 (GB), 95 (€), 7 (soporte)] -> Probablemente SÍ abandona
X_test_raw = np.array([
    [25, 2, 10, 55, 1],
    [30, 5, 60, 95, 7]
])

# Aplicar la misma normalización
X_test = X_test_raw / X_raw.max(axis=0)

# --- Predicción final ---
# Hacemos solo el Forward Pass
z1_test = np.dot(X_test, w1) + b1
a1_test = sigmoid(z1_test)
z2_test = np.dot(a1_test, w2) + b2
a_test = sigmoid(z2_test)

print("\n--- Prueba con Nuevos Clientes ---")
print("Probabilidad de abandono (Churn):")
print(np.round(a_test, 2))

print("\nInterpretación:")
print(f"Cliente A: {a_test[0,0]:.2f} ({(a_test[0,0]*100):.0f}%) de probabilidad de abandono.")
print(f"Cliente B: {a_test[1,0]:.2f} ({(a_test[1,0]*100):.0f}%) de probabilidad de abandono.")